# AIP Bench — Demo Notebook

Benchmarking suite for LLM hallucination detection, inference efficiency, and QA compression.

In [ ]:
from aip.bench import run_benchmark, compare, BenchmarkResult
from aip.bench import auroc_score, f1_score, exact_match, token_efficiency
from aip.bench import bootstrap_ci, paired_permutation_test
import numpy as np

## 1. Run a Single Benchmark (Synthetic Data)

In [ ]:
# Hallucination detection benchmark
result = run_benchmark('halueval')
print(result.summary())
print(f"\nAUROC: {result.metrics['auroc']:.3f}")
print(f"F1:    {result.metrics['f1']:.3f}")

## 2. Run Multiple Benchmarks

In [ ]:
results = {}
for name in ['halueval', 'ockbench', 'qa_compression']:
    results[name] = run_benchmark(name)
    print(f"{name:20s} → {results[name]}")

## 3. Compare Compression Configurations

In [ ]:
report = compare(
    configs={
        "no_pruning": {"prune_ratio": 0.0},
        "prune_25": {"prune_ratio": 0.25},
        "prune_50": {"prune_ratio": 0.50},
    },
    benchmarks=["ockbench"],
)
print(report.table())
print("\nDeltas vs no_pruning:")
print(report.deltas("no_pruning"))

## 4. Metrics Deep Dive

In [ ]:
# AUROC with bootstrap confidence interval
y_true = np.array([0, 0, 0, 1, 1, 1, 0, 1, 0, 1] * 20)
y_scores = np.array([0.1, 0.2, 0.3, 0.8, 0.9, 0.7, 0.4, 0.6, 0.35, 0.85] * 20)

auroc = auroc_score(y_true, y_scores)
ci = bootstrap_ci(auroc_score, y_true, y_scores, n_bootstrap=2000)

print(f"AUROC: {auroc:.4f}")
print(f"95% CI: [{ci['ci_low']:.4f}, {ci['ci_high']:.4f}]")

## 5. Token Efficiency Analysis

In [ ]:
# Compare different compression levels
for prune in [0.0, 0.10, 0.25, 0.50]:
    result = run_benchmark('ockbench', prune_ratio=prune)
    m = result.metrics
    print(f"Prune {prune:.0%}: accuracy={m['accuracy']:.2f}, "
          f"savings={m['savings_ratio']:.1%}, "
          f"efficiency={m['efficiency']:.3f}")

## 6. Export Results

In [ ]:
from aip.bench import to_json, to_csv, to_html

# Export single result
result = run_benchmark('halueval')
print("JSON:", to_json(result)[:200], "...")
print("\nCSV:", to_csv(result)[:200], "...")

## 7. Available Benchmarks and Datasets

In [ ]:
from aip.bench import list_datasets

datasets = list_datasets()
for name, info in sorted(datasets.items()):
    print(f"  {name:30s} [{info['category']:15s}] {info['description']}")